# 第四章：符号分类

两种典型分类思路：

1. **k 近邻分类（k-NN）**：找最像的邻居投票（案例：作曲家识别）
2. **支持向量机（SVM）**：划一条最大间隔的界（案例：音乐风格分类）

最后把本地中国曲目 MIDI 输入由西方艺术家目录训练的分类器，考察闭集模型跨目录应用时的输出与标签空间限制。

数据来源：
- 西方作曲家：`CODE/datasets/lmd_clean_midi/`
- 本地中国曲目 MIDI：`CODE/datasets/melodies/`

图片输出：`CODE/chapter04/output_figures/`（600 dpi）

## 0. 环境与配置

In [ ]:
import os
import random
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import pretty_midi
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# 中文字体
plt.rcParams['font.sans-serif'] = [
    'PingFang SC', 'Hiragino Sans GB', 'Microsoft YaHei', 'SimHei',
    'Arial Unicode MS', 'Noto Sans CJK SC', 'DejaVu Sans',
]
plt.rcParams['axes.unicode_minus'] = False
# 强制白底
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['axes.edgecolor'] = 'black'
plt.rcParams['axes.labelcolor'] = 'black'
plt.rcParams['xtick.color'] = 'black'
plt.rcParams['ytick.color'] = 'black'
plt.rcParams['text.color'] = 'black'

# 随机种子
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# 路径推断：从 cwd 向上找含 CODE/datasets 的目录
_p = os.getcwd()
while not os.path.exists(os.path.join(_p, 'CODE', 'datasets')):
    _parent = os.path.dirname(_p)
    if _parent == _p:
        raise FileNotFoundError("未找到项目根目录（包含 CODE/datasets 的目录），请在项目内运行本 Notebook")
    _p = _parent
BASE_DIR = _p

DATASET_DIR = os.path.join(BASE_DIR, 'CODE', 'datasets')
LMD_DIR = os.path.join(DATASET_DIR, 'lmd_clean_midi')
MELODY_DIR = os.path.join(DATASET_DIR, 'melodies')
FIGURES_DIR = os.path.join(BASE_DIR, 'CODE', 'chapter04', 'output_figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

print('BASE_DIR:', BASE_DIR)
print('FIGURES_DIR:', FIGURES_DIR)


## 1. 工具函数：MIDI 解析与特征提取

In [ ]:
def load_midi_notes(filepath, skip_drums=True):
    """按 MIDI tick 聚合同 onset 音符，并取最高音作为顶线启发式。"""
    pm = pretty_midi.PrettyMIDI(filepath)
    onset_groups = {}
    for instrument in pm.instruments:
        if skip_drums and instrument.is_drum:
            continue
        for note in instrument.notes:
            onset_tick = int(round(pm.time_to_tick(note.start)))
            onset_groups.setdefault(onset_tick, []).append(note.pitch)
    return [(max(onset_groups[tick]), tick) for tick in sorted(onset_groups)]


def extract_feature_vector(notes):
    """37 维特征：12 维音级直方图（PCH）+ 25 维音程直方图（-12..+12）。"""
    if len(notes) < 2:
        return None
    pitches = [p for p, _ in notes]

    pitch_class_hist = np.zeros(12)
    for p in pitches:
        pitch_class_hist[p % 12] += 1
    if pitch_class_hist.sum() > 0:
        pitch_class_hist /= pitch_class_hist.sum()

    interval_hist = np.zeros(25)
    for i in range(1, len(pitches)):
        d = pitches[i] - pitches[i - 1]
        if -12 <= d <= 12:
            interval_hist[d + 12] += 1
    # 与正文定义一致：分母是全部 N-1 个相邻音程。超出 [-12, 12]
    # 的跳进不进入任一 bin，因此 25 个 bin 的和可能小于 1。
    interval_hist /= len(pitches) - 1

    return np.concatenate([pitch_class_hist, interval_hist])


def two_directional_features(notes):
    """二维示意特征：(平均音程绝对值, 上行音程占比)，仅用于 k-NN 示意图。"""
    if len(notes) < 2:
        return None
    pitches = [p for p, _ in notes]
    intervals = [pitches[i+1] - pitches[i] for i in range(len(pitches)-1)]
    if not intervals:
        return None
    avg_abs = float(np.mean([abs(d) for d in intervals]))
    up_ratio = sum(1 for d in intervals if d > 0) / len(intervals)
    return np.array([avg_abs, up_ratio])


def load_corpus(directory, max_files=None, min_notes=10):
    """从某目录批量加载 MIDI，提取 37 维特征矩阵。"""
    if not os.path.exists(directory):
        return np.zeros((0, 37)), []
    feature_list, file_list = [], []
    files = sorted(os.listdir(directory))
    if max_files is not None:
        files = files[:max_files]
    skipped = 0  # 统计解析失败被跳过的文件数，避免静默丢弃
    for filename in files:
        if not filename.lower().endswith(('.mid', '.midi')):
            continue
        filepath = os.path.join(directory, filename)
        try:
            notes = load_midi_notes(filepath)
            if len(notes) < min_notes:
                continue
            feat = extract_feature_vector(notes)
            if feat is None:
                continue
            feature_list.append(feat)
            file_list.append(filename)
        except Exception:
            skipped += 1
            continue
    if skipped:
        print(f'提示：目录 {os.path.basename(directory)} 中 {skipped} 个 MIDI 文件解析失败，已跳过')
    return np.array(feature_list), file_list


## 2. k 近邻分类（k-NN）：作曲家识别

**案例**：巴赫 / 贝多芬 / 肖邦三个本地目录标签的分类。

### 2.1 数据加载

每个目录随机抽 15 个文件（取最少类的数量做平衡），提取 37 维特征。复音 MIDI 先按 tick 聚合同 onset 音符并取最高音，形成可复核的顶线启发式；它避免把和弦内文件顺序误当旋律音程，却不等于人工声部分离。由于本地数据缺少来源映射，且随机按文件切分可能让同曲不同版本跨越训练集与测试集，结果不能外推为一般的作曲家识别性能。


In [ ]:
COMPOSERS = {
    '巴赫':   'Bach Johann Sebastian',
    '贝多芬': 'Ludwig van Beethoven',
    '肖邦':   'Chopin Frederic',
}
SAMPLES_PER_COMPOSER = 15  # 平衡到最少类的数量

composer_features = {}
for label, dirname in COMPOSERS.items():
    full_dir = os.path.join(LMD_DIR, dirname)
    X_all, files = load_corpus(full_dir)
    if len(X_all) > SAMPLES_PER_COMPOSER:
        idx = np.random.choice(len(X_all), SAMPLES_PER_COMPOSER, replace=False)
        X_all = X_all[idx]
    composer_features[label] = X_all
    print(f'{label}: 共 {len(X_all)} 首特征向量')

X_composer = np.vstack([composer_features[c] for c in COMPOSERS])
y_composer = np.array([c for c in COMPOSERS for _ in composer_features[c]])
print(f'\n合并后：X.shape={X_composer.shape}, y 类别={np.unique(y_composer, return_counts=True)}')


### 2.2 k=3 邻居投票示意图

示意图仅使用两个手工选取的二维特征：
- 横轴：**平均音程绝对值**（相邻音的平均移动幅度）
- 纵轴：**上行音程占比**（旋律方向倾向）

> 该图只说明投票机制，不表示 37 维特征空间中的实际分类边界。

**正文图题：$k=3$ 邻居投票示意图。**


In [ ]:
def collect_2d_features(directory, n_samples):
    feats = []
    if not os.path.exists(directory):
        return np.zeros((0, 2))
    for filename in sorted(os.listdir(directory)):
        if not filename.lower().endswith(('.mid', '.midi')):
            continue
        try:
            notes = load_midi_notes(os.path.join(directory, filename))
            if len(notes) < 10:
                continue
            f = two_directional_features(notes)
            if f is not None:
                feats.append(f)
        except Exception:
            continue
    feats = np.array(feats)
    if len(feats) > n_samples:
        idx = np.random.choice(len(feats), n_samples, replace=False)
        feats = feats[idx]
    return feats


composer_features_2d = {
    label: collect_2d_features(os.path.join(LMD_DIR, dirname), SAMPLES_PER_COMPOSER)
    for label, dirname in COMPOSERS.items()
}
for label, pts in composer_features_2d.items():
    print(f'{label}: 2D 散点 {len(pts)} 个，'
          f'均值=({pts[:,0].mean():.2f}, {pts[:,1].mean():.2f})')

# 待分类点位于三类点的几何中心附近，因此三类都会参与近邻投票。
all_pts = np.vstack(list(composer_features_2d.values()))
query_point = all_pts.mean(axis=0)
print(f'\n查询点 (待分类)：({query_point[0]:.2f}, {query_point[1]:.2f})')

# 找 k=3 个最近邻
all_labels = np.concatenate([
    [label] * len(pts) for label, pts in composer_features_2d.items()
])
distances = np.linalg.norm(all_pts - query_point, axis=1)
nearest_3_idx = np.argsort(distances)[:3]
nearest_labels = all_labels[nearest_3_idx]
votes = Counter(nearest_labels)
predicted = votes.most_common(1)[0][0]
print(f'最近 3 邻居标签：{list(nearest_labels)}  →  投票结果：{predicted}')

# 绘图
fig, ax = plt.subplots(figsize=(8, 6))
markers = {
    '巴赫':   ('o', 'black',     'black'),
    '贝多芬': ('s', '#444444',   '#888888'),
    '肖邦':   ('^', '#888888',   'white'),
}
for label, pts in composer_features_2d.items():
    m, edge, fill = markers[label]
    ax.scatter(pts[:, 0], pts[:, 1], marker=m, s=90,
               edgecolor=edge, facecolor=fill, label=label,
               linewidth=1.4, alpha=0.9)

# 待分类点
ax.scatter(query_point[0], query_point[1], marker='X', s=280,
           edgecolor='red', facecolor='red', label='待分类样本', zorder=5)

# 邻居连线 + 红圈
for idx in nearest_3_idx:
    ax.plot([query_point[0], all_pts[idx, 0]],
            [query_point[1], all_pts[idx, 1]],
            color='red', linestyle='--', lw=1.2, alpha=0.7)
    ax.scatter(all_pts[idx, 0], all_pts[idx, 1],
               marker='o', s=240, edgecolor='red',
               facecolor='none', lw=2.2, zorder=4)

ax.set_xlabel('平均音程绝对值（半音）', fontsize=12)
ax.set_ylabel('上行音程占比', fontsize=12)
ax.legend(loc='upper left', bbox_to_anchor=(0.02, 0.98), fontsize=10)
ax.grid(True, linestyle=':', alpha=0.5)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig_knn_vote.png'), dpi=600, bbox_inches='tight')
plt.show()
print('\nsaved → fig_knn_vote.png')


### 2.3 k-NN 三分类训练与评估

使用全部 37 维特征。先做一次文件级分层训练/测试切分；仅在训练集内部用 3 折分层交叉验证比较 k=1, 3, 5, 7，再用选定的 k 一次性评估保留测试集。样本仍很少，且没有重复外层切分或显著性检验。


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_composer, y_composer, test_size=0.2, random_state=SEED, stratify=y_composer
)

inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
print('训练集内部 3 折交叉验证准确率：')
best_k, best_cv_acc = 1, -np.inf
for k in [1, 3, 5, 7]:
    knn = KNeighborsClassifier(n_neighbors=k)
    fold_scores = cross_val_score(knn, X_train, y_train, cv=inner_cv, scoring='accuracy')
    mean_score = float(fold_scores.mean())
    print(f'  k={k}: mean={mean_score:.3f}, folds={np.round(fold_scores, 3).tolist()}')
    if mean_score > best_cv_acc:
        best_cv_acc, best_k = mean_score, k

composer_classifier = KNeighborsClassifier(n_neighbors=best_k)
composer_classifier.fit(X_train, y_train)
predicted_composers = composer_classifier.predict(X_test)
test_acc = accuracy_score(y_test, predicted_composers)
print(f'\n训练集内部选定 k = {best_k}（CV mean={best_cv_acc:.3f}）')
print(f'保留测试集准确率 = {test_acc:.3f}')

labels = list(COMPOSERS.keys())
cm = confusion_matrix(y_test, predicted_composers, labels=labels)
print('\n混淆矩阵（行 = 真实，列 = 预测）：')
print('              ' + '  '.join(f'{c:>6}' for c in labels))
for i, c in enumerate(labels):
    print(f'真实 = {c:<5} ' + '  '.join(f'{v:>6d}' for v in cm[i]))


## 3. 支持向量机（SVM）：音乐风格分类

**案例**：古典 / 流行 / 爵士三类风格分类。

### 3.1 数据加载

| 风格 | 来源（lmd_clean_midi 目录） |
|---|---|
| 古典 | Bach、Beethoven、Chopin、Debussy |
| 流行 | The Beatles、ABBA、Queen |
| 爵士 | Duke Ellington、Ellington、Louis Armstrong、Frank Sinatra |

每类抽样 30 首做平衡。


In [ ]:
STYLES = {
    '古典': ['Bach Johann Sebastian', 'Ludwig van Beethoven',
             'Chopin Frederic', 'Claude Debussy'],
    '流行': ['The Beatles', 'ABBA', 'Queen'],
    '爵士': ['Duke Ellington', 'Ellington', 'Louis Armstrong', 'Frank Sinatra'],
}
SAMPLES_PER_STYLE = 30

style_features = {}
for style, dirs in STYLES.items():
    feats_pool = []
    for dirname in dirs:
        X_part, _ = load_corpus(os.path.join(LMD_DIR, dirname))
        if len(X_part) > 0:
            feats_pool.append(X_part)
    feats_pool = np.vstack(feats_pool) if feats_pool else np.zeros((0, 37))
    if len(feats_pool) > SAMPLES_PER_STYLE:
        idx = np.random.choice(len(feats_pool), SAMPLES_PER_STYLE, replace=False)
        feats_pool = feats_pool[idx]
    style_features[style] = feats_pool
    print(f'{style}: 抽样 {len(feats_pool)} 首')

X_style = np.vstack([style_features[s] for s in STYLES])
y_style = np.array([s for s in STYLES for _ in style_features[s]])
print(f'\n合并后：X.shape={X_style.shape}')


### 3.2 最大间隔示意图（古典 vs 流行）

仅取两类（古典 vs 流行）做线性 SVM 二分类。37 维特征经 PCA 压到 2 维以可视化决策边界、margin 与支持向量。

> 图中只显示 PCA 的二维投影；后面的多类分类仍在原 37 维空间用 RBF 核完成。

**正文图题：SVM 最大间隔示意图（古典 vs 流行；37 维特征经主成分分析投影到 2 维）。**


In [ ]:
mask = (y_style == '古典') | (y_style == '流行')
X_two = X_style[mask]
y_two = y_style[mask]

# 标准化 + PCA
scaler_demo = StandardScaler()
X_two_scaled = scaler_demo.fit_transform(X_two)
pca = PCA(n_components=2, random_state=SEED)
X_two_2d = pca.fit_transform(X_two_scaled)

# 二分类：古典 = 1, 流行 = -1
y_two_bin = np.where(y_two == '古典', 1, -1)

svm_demo = SVC(kernel='linear', C=1.0)
svm_demo.fit(X_two_2d, y_two_bin)

# 绘图
fig, ax = plt.subplots(figsize=(8, 6.2))
plot_kwargs = {
    '古典': dict(marker='o', s=90, edgecolor='black',  facecolor='black',   alpha=0.85),
    '流行': dict(marker='s', s=90, edgecolor='#444444', facecolor='#bbbbbb', alpha=0.85),
}
for cls in ['古典', '流行']:
    pts = X_two_2d[y_two == cls]
    ax.scatter(pts[:, 0], pts[:, 1], label=cls, linewidth=1.4, **plot_kwargs[cls])

# 决策边界 + margin
xlim = ax.get_xlim(); ylim = ax.get_ylim()
xx, yy = np.meshgrid(np.linspace(xlim[0], xlim[1], 300),
                     np.linspace(ylim[0], ylim[1], 300))
Z = svm_demo.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
ax.contour(xx, yy, Z, levels=[-1, 0, 1], colors='red',
           linestyles=['--', '-', '--'], linewidths=[1.2, 2.2, 1.2])

# 支持向量
sv = svm_demo.support_vectors_
ax.scatter(sv[:, 0], sv[:, 1], s=240, facecolor='none',
           edgecolor='red', lw=2.2, label='支持向量', zorder=5)

# 文字标注
ax.text(xlim[1]*0.98, -0.8, '决策超平面 $\\mathbf{w}^\\top\\mathbf{x}+b=0$',
        ha='right', va='bottom', fontsize=13, color='red')

ax.set_xlim(xlim); ax.set_ylim(ylim)
ax.set_xlabel('主成分 1', fontsize=12)
ax.set_ylabel('主成分 2', fontsize=12)
ax.legend(loc='lower left', bbox_to_anchor=(0.0, 0.0), fontsize=11)
ax.grid(True, linestyle=':', alpha=0.5)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig_svm_margin.png'), dpi=600, bbox_inches='tight')
plt.show()
print(f'\n支持向量数量：{len(sv)} / {len(X_two_2d)}')
print('saved → fig_svm_margin.png')


### 3.3 三类 SVM 训练与评估

在原 37 维空间训练 RBF 核 SVM。scikit-learn 的 `SVC` 内部按 one-vs-one 训练多类分类器；默认 `decision_function_shape='ovr'` 只把决策函数整理成 one-vs-rest 形状，并不改变底层训练策略。`probability=True` 的多类概率由两两类别概率估计再经耦合得到，不等同于对单一多类分数做一次 Platt scaling。


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_style, y_style, test_size=0.2, random_state=SEED, stratify=y_style
)
scaler_full = StandardScaler()
X_train_scaled = scaler_full.fit_transform(X_train)
X_test_scaled  = scaler_full.transform(X_test)

style_classifier = SVC(kernel='rbf', C=1.0,
                       decision_function_shape='ovr', probability=True,
                       random_state=SEED)
style_classifier.fit(X_train_scaled, y_train)
predicted_styles = style_classifier.predict(X_test_scaled)
acc = accuracy_score(y_test, predicted_styles)
print(f'测试集准确率：{acc:.3f}')

labels = list(STYLES.keys())
cm = confusion_matrix(y_test, predicted_styles, labels=labels)
print('\n混淆矩阵（行 = 真实，列 = 预测）：')
print('             ' + '  '.join(f'{c:>6}' for c in labels))
for i, c in enumerate(labels):
    print(f'真实 = {c:<3} ' + '  '.join(f'{v:>6d}' for v in cm[i]))


## 4. 案例：《茉莉花》会被分到哪一类？

把 3.3 训练好的"古典/流行/爵士"三类 SVM 拿来，输入几个本地中国曲目 MIDI 的 37 维特征，观察闭集分类器跨目录应用时的输出。

> 注意：训练标签来自本地目录，分类器只有"古典/流行/爵士"三类，没有"未知/其他"或拒识选项；这些中国曲目文件又没有来源与真值风格标签。跨目录应用可能存在分布偏移，但不能仅凭目录和预测结果证明样本属于统计意义上的分布外。强制标签和 `predict_proba` 数值都不能当作分布外检测结论。


In [ ]:
chinese_test_files = [
    '茉莉花.midi',
    '步步高.midi',
    '在那遥远的地方.midi',
    '瑶族舞曲.midi',
    '苏武牧羊.midi',
]

print(f'{"本地中国曲目":<22}{"预测风格":<10}{"古典":>8}{"流行":>8}{"爵士":>8}')
print('-' * 56)
for filename in chinese_test_files:
    filepath = os.path.join(MELODY_DIR, filename)
    if not os.path.exists(filepath):
        print(f'{filename:<22}（文件不存在，跳过）')
        continue
    notes = load_midi_notes(filepath)
    feat = extract_feature_vector(notes)
    if feat is None:
        print(f'{filename:<22}（音符过少，跳过）')
        continue
    feat_scaled = scaler_full.transform(feat.reshape(1, -1))
    pred = style_classifier.predict(feat_scaled)[0]
    proba = style_classifier.predict_proba(feat_scaled)[0]
    proba_dict = dict(zip(style_classifier.classes_, proba))
    name = os.path.splitext(filename)[0]
    print(f'{name:<22}{pred:<10}'
          f'{proba_dict.get("古典", 0):>8.3f}'
          f'{proba_dict.get("流行", 0):>8.3f}'
          f'{proba_dict.get("爵士", 0):>8.3f}')

print('这些是没有拒识选项的闭集 SVC 给出的标签，不能据此判断真实类别或 OOD 身份。')
print('表中概率是在当前训练与校准设置下，对三个已知标签的耦合概率估计。')
